# Track 09 — GraphRAG and Knowledge Graphs
### Mastering Entity-Relation Extraction, Property Graphs, Louvain Community Clustering, and Hierarchical Map-Reduce Retrieval

In standard Retrieval-Augmented Generation (**Vector RAG**), documents are segmented into independent chunk vectors and indexed in an approximate nearest neighbor space (such as HNSW). While Vector RAG excels at retrieving localized factual passages (*"What is the dimension of the embedding vector?"*), it suffers from two catastrophic architectural failure modes:

1. **Multi-Hop Relational Blindness**: Chunks reside in isolated vector coordinates. If Concept A connects to Concept B in Document 1, and Concept B enables Concept C in Document 3, Vector RAG cannot follow the relational chain `A -> B -> C`.
2. **Global Sensemaking Failure**: When asked macro-level exploratory questions (*"What are the central themes across all research papers?"* or *"How do the architectural components interact across the entire system?"*), Vector RAG fails because there is no single chunk that contains the answer. Top-$k$ chunk sampling retrieves arbitrary, fragmented passages that miss the overarching narrative.

**GraphRAG** (originated by Microsoft Research and production knowledge graph engineering) resolves these limitations by converting unstructured text into a structured, interconnected **Property Knowledge Graph**, clustering entity subgraphs into hierarchical **Communities**, and synthesizing multi-level **Community Summaries**.

```
   ┌─────────────────────────────────────────────────────────────────────────────┐
   │                       GraphRAG Dual-Search Taxonomy                         │
   ├──────────────────────────────────────┬──────────────────────────────────────┤
   │             LOCAL SEARCH             │            GLOBAL SEARCH             │
   │        (Entity-Centric Traversal)    │        (Hierarchical Map-Reduce)     │
   ├──────────────────────────────────────┼──────────────────────────────────────┤
   │ Query: "How does Transformer         │ Query: "What are the main system    │
   │        use Attention?"              │        architectures and trade-offs?"│
   │                                      │                                      │
   │ 1. Match seed entity in KG           │ 1. Scan precomputed Community        │
   │ 2. Traverse 1-to-2 hop ego-network   │    summaries (Louvain partitions)    │
   │ 3. Extract incident relational facts │ 2. Map Phase: Score community        │
   │ 4. Ground with source chunk IDs      │    relevance & extract insights      │
   │ 5. Synthesize grounded factual proof │ 3. Reduce Phase: Aggregate and rank  │
   │                                      │    macro executive summary           │
   └──────────────────────────────────────┴──────────────────────────────────────┘
```

---

### Core Learning Objectives
In this notebook, you will build and evaluate:
1. **Information Extraction**: Extracting entities, types, and relational triples `(Subject, Predicate, Object)` from text chunks with provenance tracking.
2. **Property Knowledge Graph**: Assembling directed property graphs using `networkx`, tracking edge weights, multi-chunk citations, and exporting to Neo4j Cypher statements.
3. **Hierarchical Community Detection**: Partitioning the graph using the **Louvain modularity algorithm** ($Q$) to detect cohesive topical clusters.
4. **Community Summarization (Map-Reduce)**: Synthesizing structured community reports (title, narrative summary, key findings, and significance weight).
5. **Local Search**: Navigating entity ego-networks for precise multi-hop question answering.
6. **Global Search**: Executing dataset-wide sensemaking via Map-Reduce summarization.
7. **Hybrid Retrieval**: Fusing dense vector similarity with graph relational triples.
8. **Topological Evaluation**: Measuring graph density, modularity $Q$, connected components, and answer grounding.


## 1. Setup & Environment Configuration

In this step, we configure workspace path resolution, verify our virtual environment, and import the modular GraphRAG engineering primitives from `common.graph_rag`.

All components run **100% offline on standard CPU** without requiring external databases or paid API keys.


In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root
project_root = Path.cwd()
while project_root.name != "RAG_from_scratch" and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from common.graph_rag import (
    Entity,
    Relationship,
    Community,
    GraphSearchResult,
    EntityRelationExtractor,
    KnowledgeGraphStore,
    CommunityDetector,
    CommunitySummarizer,
    GraphRAGEngine,
    GraphEvaluator,
    GraphVisualizer,
)
from common.config import settings

print(f"Project root resolved: {project_root}")
print(f"Chroma Persist Directory: {settings.chroma_persist_directory}")
print("GraphRAG modules loaded successfully!")


## 2. Visualizing GraphRAG Architecture: Vector RAG vs Local vs Global Search

Before diving into code, let us inspect the fundamental architectural distinctions between **Naive Vector RAG**, **Local GraphRAG**, and **Global GraphRAG**.

- **Naive Vector RAG**: Chunks are embedded into dense coordinates. Only isolated passages with high cosine similarity are retrieved.
- **Local GraphRAG**: Seed entities are identified in the query. The system extracts the entity's 1-hop and 2-hop ego-network along with incident relational triples, providing precise provenance and multi-hop reasoning.
- **Global GraphRAG**: Pre-clusters the entire knowledge base using Louvain community detection. When a thematic question is posed, it uses Map-Reduce to rate community insights and synthesize an overarching executive summary.


In [ ]:
from IPython.display import Image, display

arch_diagram_path = project_root / "assets" / "diagrams" / "track09_graphrag_architecture.png"
if not arch_diagram_path.exists():
    GraphVisualizer.plot_graphrag_architecture(str(arch_diagram_path))

display(Image(filename=str(arch_diagram_path)))


## 3. Information Extraction: Entities, Types, and Relational Triples

The foundation of any Knowledge Graph is **Information Extraction (IE)**. Given an unstructured text chunk, the extractor must identify:
1. **Entities ($E$)**: Canonical concept mentions (e.g., `Transformer`, `Attention`, `ChromaDB`), normalized into consistent lowercase IDs.
2. **Entity Types**: Semantic categorization (`ARCHITECTURE`, `MECHANISM`, `VECTOR_DATABASE`, `FRAMEWORK`).
3. **Relationships ($R$)**: Directed subject-predicate-object triples `(Source, Relation_Type, Target)` describing factual interactions (e.g., `(Transformer, USES, Attention)`).
4. **Source Chunk Citations**: Strict provenance linking each node and edge back to the specific chunk ID from which it originated.

In this step, we demonstrate the deterministic `EntityRelationExtractor`, which executes pattern matching over a domain ontology.


In [ ]:
extractor = EntityRelationExtractor()

sample_chunk_text = (
    "The Transformer architecture relies on Attention. Self-Attention implements Multi-Head Attention. "
    "Encoder consists of Self-Attention. ChromaDB stores Embeddings and utilizes HNSW for indexing."
)

entities, relationships = extractor.extract_from_text(sample_chunk_text, chunk_id="doc_chunk_1")

print(f"Extracted {len(entities)} Entities:")
for e in entities:
    print(f"  - [{e.entity_type}] {e.name} (ID: {e.id}, Sources: {e.source_chunks})")

print(f"\nExtracted {len(relationships)} Relational Triples:")
for r in relationships:
    print(f"  - [{r.source.title()}] --({r.relation_type})--> [{r.target.title()}] | Desc: {r.description}")


## 4. Property Knowledge Graph Construction & Neo4j Cypher Interoperability

Once triples are extracted, they are ingested into a **Property Knowledge Graph**. The graph store must handle:
- **Node Property Merging**: When multiple chunks mention the same entity, the graph unions their source chunk citations and updates the entity description without creating duplicate nodes.
- **Edge Weight Accumulation**: Repeated co-occurrences strengthen the relationship weight $w_{ij}$, reflecting higher relational confidence.
- **Graph Traversal Primitives**: Shortest path detection, $k$-hop ego-network retrieval, and PageRank centrality ranking.
- **Neo4j Cypher Export**: Translating the in-memory graph into declarative Cypher queries (`MERGE (e:Entity ...)`, `MERGE (s)-[:RELATION]->(t)`) for enterprise database integration.


In [ ]:
kg_store = KnowledgeGraphStore()

# Populate the Knowledge Graph with extracted entities and relations
for ent in entities:
    kg_store.add_entity(ent)
for rel in relationships:
    kg_store.add_relationship(rel)

print(f"Total Nodes in KG: {kg_store.graph.number_of_nodes()}")
print(f"Total Edges in KG: {kg_store.graph.number_of_edges()}")

# Compute PageRank Centrality
pagerank = kg_store.compute_centrality()
print("\nTop Entities by PageRank Centrality:")
sorted_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)
for ent_id, score in sorted_pr[:4]:
    ent = kg_store.get_entity(ent_id)
    print(f"  - {ent.name} ({ent.entity_type}): PageRank = {score:.4f}")

# Generate Cypher Statements for Neo4j
cypher_stmts = kg_store.to_cypher_statements()
print(f"\nGenerated {len(cypher_stmts)} Neo4j Cypher Statements (First 3):")
for stmt in cypher_stmts[:3]:
    print(f"  CYPHER > {stmt}")


## 5. Hierarchical Community Detection via Louvain Optimization

In GraphRAG, raw graphs can contain thousands of nodes. Querying all nodes simultaneously is computationally intractable and floods the LLM context window.

To organize the knowledge base, GraphRAG employs **Community Detection** using the **Louvain modularity algorithm**. The algorithm partitions the graph into non-overlapping clusters $C = \{c_1, c_2, \dots, c_k\}$ by maximizing the **Newman-Girvan Modularity ($Q$)**:

$$Q = \sum_{c \in C} \left[ \frac{e_c}{2m} - \left( \frac{d_c}{2m} \right)^2 \right]$$

Where:
- $e_c$ is the number of edges entirely inside community $c$.
- $d_c$ is the sum of degrees of all nodes in community $c$.
- $m$ is the total number of edges in the network.
- $Q \in [-0.5, 1.0]$. A score of $Q > 0.3$ indicates strong community structure with significantly more internal connections than random expectation.

In this step, we detect communities on a multi-domain knowledge graph.


In [ ]:
# Expand KG with diverse multi-domain documents
corpus = [
    {"id": "c1", "text": "The Transformer architecture relies on Attention. Self-Attention implements Multi-Head Attention."},
    {"id": "c2", "text": "Encoder consists of Self-Attention. Decoder consists of Self-Attention."},
    {"id": "c3", "text": "ChromaDB stores Embeddings. ChromaDB utilizes HNSW for approximate nearest neighbors."},
    {"id": "c4", "text": "LlamaIndex indexes Embeddings and integrates ChromaDB."},
    {"id": "c5", "text": "GraphRAG partitions Knowledge Graph using Louvain algorithm. GraphRAG uses Neo4j and Cypher."},
    {"id": "c6", "text": "LangGraph orchestrates Multi-Agent RAG and utilizes LangChain."},
]

full_kg = KnowledgeGraphStore()
for doc in corpus:
    ents, rels = extractor.extract_from_text(doc["text"], doc["id"])
    for e in ents:
        full_kg.add_entity(e)
    for r in rels:
        full_kg.add_relationship(r)

detector = CommunityDetector(resolution=1.0, random_state=42)
communities, modularity_q = detector.detect_communities(full_kg)

print(f"Detected {len(communities)} distinct Louvain communities.")
print(f"Graph Modularity Q: {modularity_q:.4f} (Q > 0.3 confirms cohesive modular clustering)")

for cid, members in enumerate(communities):
    member_names = [full_kg.get_entity(m).name for m in members if full_kg.get_entity(m)]
    print(f"  Cluster {cid} ({len(members)} nodes): {', '.join(member_names)}")


## 6. Community Summarization: Hierarchical Map-Reduce

Once communities are partitioned, GraphRAG executes **Community Summarization** to convert topological graph clusters into human-readable and LLM-digestible narratives:

1. **Map Step**: For each detected community, the engine identifies the highest-degree hub entities, extracts internal relationship propositions, and writes a structured **Community Report**:
   - `Title`: Descriptive thematic label.
   - `Summary`: Cohesive paragraph detailing what the community represents.
   - `Findings`: Bullet-point factual propositions with relationship semantics.
   - `Weight`: Significance rating based on entity count and internal link density.
2. **Reduce Step**: For high-level sensemaking, the engine aggregates community reports into a unified meta-summary.


In [ ]:
summarizer = CommunitySummarizer()
community_reports = []

for cid, member_ids in enumerate(communities):
    report = summarizer.generate_community_report(
        community_id=cid,
        member_ids=member_ids,
        kg=full_kg,
        level=0
    )
    community_reports.append(report)

print("Generated Community Reports (Map Phase):\n" + "="*50)
for r in community_reports:
    print(f"\n[Community {r.id}] {r.title} (Weight: {r.weight:.1f}, Nodes: {len(r.entity_ids)})")
    print(f"  Summary: {r.summary}")
    print(f"  Findings: {'; '.join(r.findings[:2])}")

# Execute Hierarchical Reduce Step
meta_summary = summarizer.reduce_meta_summary(community_reports, query="Overview of system architecture")
print("\n" + "="*50 + "\n" + meta_summary)


## 7. Visualizing the Clustered Knowledge Graph

Visual inspection of the knowledge graph confirms how nodes cluster into communities.

In this step, we render the entire multi-domain knowledge graph using NetworkX's spring layout with nodes colored by their assigned Louvain community.


In [ ]:
kg_plot_path = project_root / "assets" / "diagrams" / "track09_community_graph.png"
GraphVisualizer.plot_knowledge_graph(
    full_kg,
    str(kg_plot_path),
    communities=community_reports,
    figsize=(11, 7)
)

display(Image(filename=str(kg_plot_path)))


## 8. Dual GraphRAG Search: Local Search (Entity-Centric Retrieval)

**Local Search** is optimized for queries focused on specific entities, mechanisms, or multi-hop connections.

### Operational Dataflow
1. **Seed Entity Resolution**: Match query terms against KG entities using token overlap and semantic embedding cosine similarity.
2. **Ego-Network Expansion**: Retrieve the 1-hop and 2-hop connected subgraphs around focal entities.
3. **Relational Fact Assembly**: Extract incident directed edges with relation types, descriptions, and source chunk provenance citations.
4. **Community Context Injection**: Inject the community summaries corresponding to the seed entities.
5. **Grounded Response Generation**: Synthesize an exact factual answer citing specific relational edges and chunk IDs.


In [ ]:
# Initialize full GraphRAG Engine
engine = GraphRAGEngine(kg=full_kg)
engine.index_documents(corpus)

local_query = "How does the Transformer utilize Attention and what components are involved?"
local_result = engine.local_search(local_query, top_k_entities=2, max_hops=1)

print(f"Search Mode: {local_result.search_mode.upper()}")
print(f"Grounded Entities: {local_result.grounded_entities}")
print(f"Grounded Source Chunks: {local_result.source_chunks}")
print(f"Relational Facts Found: {len(local_result.grounded_relationships)}")
print("\n" + "="*50)
print(local_result.answer)


## 9. Dual GraphRAG Search: Global Search (Map-Reduce Sensemaking)

**Global Search** is designed for corpus-wide exploratory questions where no single entity holds the answer (e.g., *"What are the primary architectural paradigms and system components across the corpus?"*).

### Operational Dataflow
1. **Community Discovery**: Retrieve all precomputed community reports.
2. **Map Phase**: Score each community summary against the global query using semantic cosine similarity and extract key findings.
3. **Filtering & Ranking**: Rank candidate claims by combined relevance and structural significance weight.
4. **Reduce Phase**: Aggregate top community insights into a coherent, structured executive summary with community citations.


In [ ]:
global_query = "What are the major technologies, indexing mechanisms, and framework architectures across all documents?"
global_result = engine.global_search(global_query, top_k_communities=4)

print(f"Search Mode: {global_result.search_mode.upper()}")
print(f"Grounded Communities: {global_result.grounded_communities}")
print(f"Intermediate Community Claims: {len(global_result.intermediate_claims)}")
print("\n" + "="*50)
print(global_result.answer)


## 10. Hybrid Retrieval: Graph Relational Paths + Dense Vector Chunks

While Knowledge Graphs excel at structured relationships, unstructured passages contain nuance, narrative context, and descriptive prose that cannot always be distilled into atomic triples.

**Hybrid GraphRAG** fuses:
1. **Structured Graph Evidence**: Local ego-network paths, incident edges, and community context.
2. **Unstructured Vector Context**: Ranked text passages retrieved via dense vector similarity.

In this step, we execute `hybrid_search` and demonstrate multi-source evidence fusion with unified chunk citations.


In [ ]:
vector_chunks = [
    {"id": "c1", "text": "The Transformer architecture relies on Attention. Self-Attention implements Multi-Head Attention."},
    {"id": "c3", "text": "ChromaDB stores Embeddings. ChromaDB utilizes HNSW for approximate nearest neighbors."},
    {"id": "c6", "text": "LangGraph orchestrates Multi-Agent RAG and utilizes LangChain."},
]

hybrid_query = "Explain how Transformer attention connects to embeddings and vector storage in ChromaDB."
hybrid_result = engine.hybrid_search(hybrid_query, vector_chunks=vector_chunks, top_k=2)

print(f"Search Mode: {hybrid_result.search_mode.upper()}")
print(f"Combined Provenance Chunks: {hybrid_result.source_chunks}")
print(f"Graph Relational Edges: {len(hybrid_result.grounded_relationships)}")
print("\n" + "="*50)
print(hybrid_result.answer)


## 11. Graph Diagnostics & Groundedness Evaluation

Deploying a Knowledge Graph in production requires continuous quality validation:
- **Graph Density ($D$)**: Measures topological connectivity $D = \frac{2|E|}{|V|(|V|-1)}$.
- **Connected Components ($C$)**: Detects whether the graph is cohesive or fragmented into disjoint islands.
- **Modularity ($Q$)**: Quantifies cluster partitioning quality.
- **Chunk Coverage**: Verifies what fraction of raw documents are represented in the KG.
- **Answer Groundedness**: Evaluates whether entities cited in the generated answer actually exist in the knowledge graph, detecting hallucinations.


In [ ]:
# 1. Topological Graph Metrics
topo_metrics = GraphEvaluator.evaluate_graph(full_kg, engine.communities)
print("Knowledge Graph Topological Diagnostics:")
for k, v in topo_metrics.items():
    print(f"  - {k}: {v}")

# 2. Answer Groundedness Check
test_answer = (
    "The Transformer implements Attention and Self-Attention. ChromaDB stores Embeddings. "
    "Meanwhile, QuantumFlux creates SuperPositions."  # Intentional hallucination entity
)

grounding = GraphEvaluator.evaluate_answer_groundedness(test_answer, full_kg)
print(f"\nGroundedness Evaluation:")
print(f"  - Is Grounded: {grounding['is_grounded']}")
print(f"  - Grounded Entities Identified in KG: {grounding['mentioned_entities']}")
print(f"  - Verified Grounded Count: {grounding['grounded_entity_count']}")


## 12. Architectural Summary & Practical Recommendations

### When to Use GraphRAG vs Vector RAG

| Dimension | Naive Vector RAG | GraphRAG (Local Search) | GraphRAG (Global Search) |
| :--- | :--- | :--- | :--- |
| **Primary Query Type** | Direct factual lookup (*"What is X?"*) | Multi-hop relational (*"How does X impact Y?"*) | Holistic sensemaking (*"What are the themes?"*) |
| **Indexing Structure** | HNSW / Flat dense vector index | Property Graph ($E, R$) with chunk citations | Louvain community hierarchy + summaries |
| **Query Mechanism** | Cosine similarity top-$k$ | Ego-network traversal + community context | Map-Reduce over community reports |
| **Context Window Cost** | Low ($k$ isolated chunks) | Moderate ($k$-hop subgraph + relations) | Moderate-to-High (Map-Reduce aggregation) |
| **Hallucination Risk** | Moderate (context gaps) | Extremely Low (explicit relational edges) | Low (community grounded summaries) |

---

### Key Takeaways
1. **Triples Ground Multi-Hop Reasoning**: Extracting explicit `(Source, Predicate, Target)` triples prevents the model from having to guess or hallucinate cross-document links.
2. **Community Detection Enables Global Sensemaking**: Partitioning via Louvain modularity $Q$ turns an unnavigable web of thousands of entities into structured, thematic clusters.
3. **Hierarchical Map-Reduce Overcomes Context Limits**: Generating community reports at index time allows global questions to be answered in seconds without passing entire corpora to the LLM.

### Next Steps in Curriculum
In **Track 10 — Multimodal RAG**, we extend retrieval beyond pure text to multi-vector image embeddings, visual document parsing (ColPali), tables, charts, and temporal audio/video alignment.
